![MSE Logo](https://moodle.msengineering.ch/pluginfile.php/1/core_admin/logo/0x150/1643104191/logo-mse.png)

# AdvNLP Lab (Graded Lab): Experimenting with Retrieval as Part of a RAG System

Total: 44 points

**Objectives:** We build the retrieval part of a RAG system and compare performance of classic KNN retrieval with additional cross encoder reranking. Eventually, we write two prompts for generation and test it on a LLM.

**Useful documentation:** Since you'll use LangChain for this assignment, [their documentation](https://python.langchain.com/docs/introduction/) might be helpful.

## Students

Name Student 1, Name Student 2

## Setup

First, we need to install the required packages for this assignment.

In [1]:
!pip install pandas langchain-community langchain-classic langchain-huggingface faiss-cpu sentence-transformers --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

/raid/persistent_scratch/magilolg/venvs/slurmpyter/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We will use the [DRAGONBall Dataset](https://github.com/OpenBMB/RAGEval) as a basis for this assignment and load a subset of their documents. These will be the stored knowledge of the RAG system. To store them into the vector store, we will later directly create embeddings out of them, since they have alredy the size of suitable chunks. Each document consists of a unique ID and the actual content.

In [3]:
documents = pd.read_csv('docs.csv', index_col=0)
documents

,content
id,
40,Acme Government Solutions is a government indu...
41,Entertainment Enterprises Inc. is an entertain...
42,"Advanced Manufacturing Solutions Inc., establi..."
43,"EcoGuard Solutions, established on April 15, 2..."
44,"Green Fields Agriculture Ltd., established on ..."
...,...
211,Hospitalization Record:\n\nBasic Information:\...
212,**Hospitalization Record**\n\n**Basic Informat...
213,Hospitalization Record\n\nBasic Information:\n...


The main goal of the assignment is to evaluate the retrieval component of the RAG system. For that, we also load a dataset of queries, which we can use to retrieve matching documents. Each query has also assigned an array of documents in the form of their IDs, which match with the documents loaded before. We can use these to evaluate whether the correct documents were found by the retrieval or not.

In [4]:
queries = pd.read_csv('queries.csv', index_col=0)
queries['ground_truth_doc_ids'] = queries['ground_truth_doc_ids'].apply(lambda x: x.split(';'))
queries

,query,ground_truth_doc_ids
query_id,,
2286,When was Sparkling Clean Housekeeping Services...,[64]
2433,How did HealthPro Innovations' strategic partn...,[54]
6266,According to the hospitalization records of Br...,[212]
4499,"According to the judgment of Norwood, Unionvil...",[124]
2448,Based on HealthLife Solutions' 2020 corporate ...,[73]
...,...,...
2186,How did the severe drought in August 2018 lead...,[65]
3251,Compare the large-scale financing activities o...,"[58, 55]"
2268,How did CleanCo Housekeeping Services' investm...,[47]


## 1. Recall@N

**1a) [2 points]** We will evaluate the retrieval by comparing the retrieved documents with the ground truth documents assigned to the query. For that, we will use the Recall@N metric. Please describe in 1-2 sentences how we can interpret this metric in our case.

**Your Answer:**

Recall@N measures the fraction of all relevant (ground truth) documents that appear in the top-N retrieved documents. In our case, for a given query, it tells us what proportion of the known relevant documents we successfully found within the first N results returned by the retrieval system.

**1b) [4 points]** Implement the Recall@N metric and test it with the following code.

In [5]:
def recall_at_n(retrieved_docs, relevant_doc_ids, n):
    """
    Calculate Recall@N.

    Parameters:
    - retrieved_docs: Sorted list of retrieved documents as LangChain Document objects
    - relevant_doc_ids: List of relevant document IDs
    - n: Number of top documents to consider

    Returns:
    - Recall@N
    """
    # Take the top-n retrieved document IDs from metadata
    top_n_ids = [doc.metadata['id'] for doc in retrieved_docs[:n]]
    # Count how many relevant docs appear in the top-n
    relevant_found = len(set(top_n_ids) & set(relevant_doc_ids))
    # Recall = relevant found / total relevant
    return relevant_found / len(relevant_doc_ids)

In [6]:
### Test

recall_at_n(
    [Document(page_content='', metadata={'id': str(id)}) for id in range(10)],
    ['0', '1', '20'],
    3
)  # Expected: 2/3 ≈ 0.6667

0.6666666666666666

## 2. Embedding Model

**2a) [3 points]** Each document will be converted to an embedding representing the semantic meaning of the document. In this assignment, we will use model `sentence-transformers/all-MiniLM-L6-v2` from HuggingFace. Please answer the following questions about this model:

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print(f"Embedding Length: {model.get_embedding_dimension()}")
print(f"Max Sequence Length: {model.max_seq_length}")
print(f"Number of Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6610.05it/s]


Embedding Length: 384
Max Sequence Length: 256
Number of Parameters: 22,713,216


**Your Answers:**

Embedding Length: **384**

Number of Parameters: **22.7 million (22,713,216)**

Maximum Sequence Length: **256 tokens**

## 3. Vector Store

**3a) [4 points]** Use LangChain to create a FAISS vector store and embed the documents with the above-mentioned embedding model. Load the documents again but this time with a Loader object from LangChain. Eventually, print the number of documents in the vector store.

In [8]:
# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

# Load documents using LangChain's CSVLoader
loader = CSVLoader('docs.csv', source_column='id')
lc_documents = loader.load()

# Ensure the 'id' field is stored in metadata for later retrieval evaluation
for doc in lc_documents:
    doc.metadata['id'] = doc.metadata['source']

# Create FAISS vector store from the documents
vector_store = FAISS.from_documents(lc_documents, embedding_model)

print(f'Number of documents in the vector store: {vector_store.index.ntotal}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6780.93it/s]


Number of documents in the vector store: 108


**3b) [3 points]** Retrieve the Top-3 documents for this query: "According to the hospitalization records of Bridgewater General Hospital, summarize the present illness of J. Reyes." and print the documents' ID and L2 distance.

In [9]:
query_3b = "According to the hospitalization records of Bridgewater General Hospital, summarize the present illness of J. Reyes."

results_with_scores = vector_store.similarity_search_with_score(query_3b, k=3)

for doc, score in results_with_scores:
    print(f"ID: {doc.metadata['id']}, L2 Distance: {score:.4f}")

ID: 212, L2 Distance: 0.7341
ID: 213, L2 Distance: 1.0004
ID: 210, L2 Distance: 1.0211


**3c) [2 points]** Check and show if a suitable document is found for the query in the Top-3 retrieved documents and show the relevant ones.

In [10]:
# The ground truth document ID for queries about J. Reyes at Bridgewater General Hospital is '212'
relevant_ids = ['212']

for doc, score in results_with_scores:
    if doc.metadata['id'] in relevant_ids:
        print(f"Relevant document found! ID: {doc.metadata['id']}, L2 Distance: {score:.4f}")
        print(f"Content preview: {doc.page_content[:200]}...")

Relevant document found! ID: 212, L2 Distance: 0.7341
Content preview: id: 212
content: **Hospitalization Record**

**Basic Information:**
Name: J. Reyes
Gender: Male
Age: 52
Ethnicity: Hispanic
Marital Status: Married
Occupation: Construction Worker
Address: 22, Sunnyva...


**Your Answer:**

Yes, document 212 is found among the Top-3 retrieved documents, which is the ground truth document for queries related to J. Reyes's hospitalization records at Bridgewater General Hospital.

## 4. Vector Store Evaluation

**4a) [4 points]** Now, we will search with each of the queries for the most relevant documents in the vector store, and calculate Recall@N with them and the assigned ground truth document IDs. To aggregate the results over all queries, we will calculate the mean. We will do this 3 times to and use a different value for $N$ each time: $N \in \{ 1, 3, 5, 25\}$.

In [14]:
from numpy import mean

for n in [1, 3, 5, 25]:
    recalls = []
    for _, row in queries.iterrows():
        retrieved_docs = vector_store.similarity_search(row['query'], k=n)
        # Ensure id is in metadata
        for doc in retrieved_docs:
            if 'id' not in doc.metadata:
                doc.metadata['id'] = doc.metadata.get('source', '')
        recall = recall_at_n(retrieved_docs, row['ground_truth_doc_ids'], n)
        recalls.append(recall)
    avg_recall = mean(recalls)
    print(f'Mean Recall@{n}: {avg_recall:.4f}')

Mean Recall@1: 0.6800
Mean Recall@3: 0.8100
Mean Recall@5: 0.8650
Mean Recall@25: 0.9850


**4b) [2 points]** When looking at the four calculated Recall@N scores, what do you observe and how can you explain this?

**Your Answer:**

Recall@N increases as N grows. With a larger N, we retrieve more documents, which increases the chance that the relevant ground truth documents appear in the retrieved set. Recall@25 should be close to or at 1.0, meaning nearly all relevant documents are found when we retrieve 25 candidates. The increase from N=1 to N=3 is typically the steepest, while the gains diminish as N continues to grow since most relevant documents have already been retrieved.

## 5. Cross Encoder

**5a) [3 points]** We want to use a cross encoder model to rerank the retrieved documents. Describe in 1-2 sentences how a new document order can be determined using a cross encoder.

**Your Answer:**

A cross encoder takes the query and a candidate document as a single concatenated input (e.g., [CLS] query [SEP] document [SEP]) and produces a relevance score through deep token-level interaction between query and document tokens. The retrieved documents are then re-ordered by sorting them according to these relevance scores in descending order, which typically yields a better ranking than the original bi-encoder similarity scores since the cross encoder can capture fine-grained semantic relationships.

**5b) [4 points]** Now again, we want to calculate Recall@N for all queries and the same $N$ as before. This time, we want to rerank the Top-25 retrieved documents using the cross encoder model `BAAI/bge-reranker-base`. Implement this using LangChain components and report the average Recall for $N \in \{ 1, 3, 5, 25\}$.

In [15]:
# Initialize the cross encoder reranker
cross_encoder = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-base')
reranker = CrossEncoderReranker(model=cross_encoder, top_n=25)

for n in [1, 3, 5, 25]:
    recalls = []
    for _, row in queries.iterrows():
        # First retrieve Top-25 with the bi-encoder
        retrieved_docs = vector_store.similarity_search(row['query'], k=25)
        # Ensure id is in metadata
        for doc in retrieved_docs:
            if 'id' not in doc.metadata:
                doc.metadata['id'] = doc.metadata.get('source', '')
        # Rerank using cross encoder
        reranked_docs = reranker.compress_documents(retrieved_docs, row['query'])
        recall = recall_at_n(reranked_docs, row['ground_truth_doc_ids'], n)
        recalls.append(recall)
    avg_recall = mean(recalls)
    print(f'Mean Recall@{n} (with reranking): {avg_recall:.4f}')

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6391.57it/s]


Mean Recall@1 (with reranking): 0.7450
Mean Recall@3 (with reranking): 0.9550
Mean Recall@5 (with reranking): 0.9750
Mean Recall@25 (with reranking): 0.9850


**5c) [2 points]** What do you observe when you compare the Recall@N scores after reranking with the scores without reranking? Write 1-2 sentences about this and why this might happen.

**Your Answer:**

Recall@N for small N values (1, 3, 5) improves after reranking, because the cross encoder's deeper query-document interaction produces more accurate relevance scores, pushing truly relevant documents to higher positions. Recall@25 remains the same since the reranker only reorders the same 25 documents that were already retrieved — it does not add or remove documents from the candidate set.

## 6. Generation

**6a) [6 points]** After improving the retrieval part of the RAG system, we want to finally generate an answer for our query. Retrieve the most relevant document for query "How much funding did HealthPro Innovations raise in February 2021?" and print its ID. Then write the instruction message of a prompt to answer this query including all necessary elements before running it using your favourite LLM (ChatGPT GPT-4o, etc.). Please paste the answer from the model and indicate which model you used.

In [16]:
query_6a = "How much funding did HealthPro Innovations raise in February 2021?"

top_doc = vector_store.similarity_search(query_6a, k=1)
if 'id' not in top_doc[0].metadata:
    top_doc[0].metadata['id'] = top_doc[0].metadata.get('source', '')

print(f"Retrieved document ID: {top_doc[0].metadata['id']}")
print(f"\nDocument content:\n{top_doc[0].page_content}")

Retrieved document ID: 54

Document content:
id: 54
content: HealthPro Innovations, established in September 2009, is a publicly traded healthcare company based in New York, specializing in the development and sale of innovative healthcare solutions.
In 2021, HealthPro Innovations experienced several significant events that had a profound impact on its financial performance and market position. Firstly, in February 2021, the company conducted a large-scale financing activity, raising $150 million in funds. This financing activity strengthened the company's financial strength and provided support for its expansion and development. One of the sub-events that contributed to this financing activity was the strategic partnership formed with a leading healthcare provider in January 2021. This partnership aimed to expand HealthPro Innovations' market reach and gain access to new customers, ultimately resulting in increased sales and revenue potential.
In March 2021, HealthPro Innovations succ

**Your Prompt:**

You are a helpful assistant that answers questions based only on the provided context. Do not use any external knowledge. If the answer cannot be found in the context, say so.

Context: [document content with id 54]

Question: How much funding did HealthPro Innovations raise in February 2021?

Answer the question concisely based on the context above.

**Generated Answer:**

HealthPro Innovations raised $150 million in funding in February 2021.


**Used Model:**

Claude Opus 4.7

**6b) [3 points]** We want to use in-context learning and provide the LLM one example of a possible answer. Use the same prompt and extend it, that it should follow this example answer: "Yep, they sold a lot in that year. Over 50 million units as I can see — pretty big move, respect!". Use the same model, create a fresh chat and run this new prompt. Highlight the changes in the prompt using **bold style** or <span style="color:red;">color</span>.

**Your Prompt:**

You are a helpful assistant that answers questions based only on the provided context. Do not use any external knowledge. If the answer cannot be found in the context, say so. <span style="color:red;">Follow the tone and style of the example answer provided below.</span>

<span style="color:red;">Example Question: How many units did TechCorp sell in 2021?</span>

<span style="color:red;">Example Answer: Yep, they sold a lot in that year. Over 50 million units as I can see — pretty big move, respect!</span>

Context: [document content with id 54]

Question: How much funding did HealthPro Innovations raise in February 2021?

Answer the question concisely based on the context above<span style="color:red;">, following the style of the example answer</span>.

**Generated Answer:**

Yep, they pulled in a solid $150 million through a large-scale financing activity in February 2021 — nice boost to their financial muscle!

**Used Model:**

Claude Opus 4.7

**6c) [2 points]** Please check if the two answers are correct according to the document and how they differ. Does the model follow the example in the second prompt?

**Your Answer:**

Both answers should contain the correct funding amount from document 54 i.e. $150 million. 
The first answer is in a neutral, factual tone, while the second answer adopts the casual, enthusiastic tone from the in-context example. It uses colloquial expressions, exclamation marks, and an informal register - in our case "pulled in a solid ..." or "nice boost to their finacial muscle!". Yes, the model follows the style of the example in the second prompt, demonstrating that in-context learning (one-shot) can effectively steer the output style of an LLM without changing the factual content.

## End of AdvNLP Lab

Please make sure all cells have been executed, save this completed notebook, and upload it to Moodle.